# Validate the assignment of plants to BAs

Checks how well OGE's assignment of plants to balancing authorities matches the boundaries that BAs report to EIA-930:
- **Fit with EIA-930:** OGE net generation by BA and fuel compared with reconciled EIA-930 data, for all generation and for the generators that EIA-930A Schedule 2 lists (i.e. that some BA includes in its EIA-930 data).
- **Map check:** plants more than 100 km from any plant in their own BA, and at least 3 times closer to a plant in another BA.

Requires the outputs of the data pipeline for `year`, including the cleaned EIA-930A tables in `outputs/{year}/eia930/`.

In [ ]:
import sys

sys.path.append("../../src")

import numpy as np
import pandas as pd
import plotly.express as px
from scipy.spatial import cKDTree

import oge.eia930a as eia930a
import oge.load_data as load_data
from oge.column_checks import get_dtypes
from oge.eia930 import energy_source_to_fuel_category_eia930
from oge.filepaths import outputs_folder
from oge.helpers import assign_fuel_category_to_esc

In [ ]:
year = 2024
# column of plant_static_attributes that assigns plants to BAs
ba_col = "ba_code"

## Functions

Loading OGE generation by generator, and EIA-930 generation by local month.

In [ ]:
def flag_generators_listed_in_eia930a(
    generators: pd.DataFrame, schedule_2: pd.DataFrame
) -> pd.DataFrame:
    """Identifies which generators are listed in EIA-930A Schedule 2, i.e. reflected in
    the EIA-930 net generation of a BA.

    A generator is matched on its plant ID and generator ID (in upper case). If the
    generator isn't listed, but other generators at its plant are, it's matched on its
    plant. These are mostly parts of combined cycle plants that a BA listed under a
    different generator ID.

    Args:
        generators (pd.DataFrame): table with "plant_id_eia" and "generator_id" columns.
        schedule_2 (pd.DataFrame): EIA-930A Schedule 2 of one year.

    Returns:
        pd.DataFrame: `generators` with a "listed_in_eia930a" column, which is
            "generator", "plant" or "unlisted".
    """
    generator_listed = pd.MultiIndex.from_arrays(
        [
            generators["plant_id_eia"],
            generators["generator_id"].astype("string").str.strip().str.upper(),
        ]
    ).isin(pd.MultiIndex.from_frame(schedule_2[["plant_id_eia", "generator_id"]]))
    plant_listed = generators["plant_id_eia"].isin(schedule_2["plant_id_eia"].dropna())
    return generators.assign(
        listed_in_eia930a=np.select(
            [generator_listed, plant_listed], ["generator", "plant"], "unlisted"
        )
    )


def load_generator_generation_for_ba_validation(year: int) -> pd.DataFrame:
    """Loads the monthly net generation of each generator from the outputs of the data
    pipeline, with the information needed to compare it with EIA-930 by BA.

    Net generation is from `eia923_allocated`, i.e. EIA-923 allocated to generators.
    Each generator is assigned the EIA-930 fuel category of its energy source, with
    pumped storage in "hydro" as in EIA-930's original energy sources. Its BAs are those
    of its plant in `plant_static_attributes`. `listed_in_eia930a` says whether the
    generator is listed in the EIA-930A Schedule 2 used for `year` (see
    `flag_generators_listed_in_eia930a`).

    Args:
        year (int): a four-digit year.

    Returns:
        pd.DataFrame: monthly net generation of each generator.
    """
    generation = pd.read_csv(
        outputs_folder(f"{year}/eia923_allocated_{year}.csv.zip"),
        usecols=[
            "report_date",
            "plant_id_eia",
            "generator_id",
            "prime_mover_code",
            "energy_source_code",
            "net_generation_mwh",
        ],
        dtype=get_dtypes(),
        parse_dates=["report_date"],
    )
    generation = assign_fuel_category_to_esc(
        generation, fuel_category_names=["fuel_category_eia930"]
    )
    generation.loc[generation["prime_mover_code"] == "PS", "fuel_category_eia930"] = (
        "hydro"
    )
    plant_attributes = pd.read_csv(
        outputs_folder(f"{year}/plant_static_attributes_{year}.csv.zip"),
        usecols=[
            "plant_id_eia",
            "plant_name_eia",
            "ba_code",
            "ba_code_physical",
            "state",
            "latitude",
            "longitude",
        ],
        dtype=get_dtypes(),
    )
    generation = generation.merge(
        plant_attributes, how="left", on="plant_id_eia", validate="m:1"
    )

    schedule_2 = eia930a.read_eia930a_tables(
        outputs_folder(f"{year}/eia930/")
    ).schedule_2
    return flag_generators_listed_in_eia930a(generation, schedule_2)


def aggregate_eia930_generation_by_month(
    eia930_generation: pd.DataFrame, year: int
) -> pd.DataFrame:
    """Aggregates hourly EIA-930 net generation by energy source to local months and
    the fuel categories of reference_tables/energy_source_groups.csv.

    Args:
        eia930_generation (pd.DataFrame): hourly EIA-930 net generation by energy
            source, e.g. the "reconciled" table written by `eia930.clean_930`.
        year (int): the year to keep, in each BA's local time.

    Returns:
        pd.DataFrame: monthly net generation by "ba_code", "report_date" and
            "fuel_category_eia930".
    """
    timezones = (
        load_data.load_ba_reference()
        .set_index("ba_code")["timezone_local"]
        .dropna()
        .to_dict()
    )
    monthly = []
    for ba, df in eia930_generation.groupby("balancing_authority_code_eia"):
        local_month = (
            df["datetime_utc"]
            .dt.tz_convert(timezones.get(ba, "UTC"))
            .dt.tz_localize(None)
            .dt.to_period("M")
            .dt.to_timestamp()
        )
        monthly.append(
            df.assign(
                ba_code=ba,
                report_date=local_month,
                # "unknown" generation is in no fuel category
                fuel_category_eia930=df["generation_energy_source"]
                .map(energy_source_to_fuel_category_eia930)
                .fillna("other"),
            )
            .loc[local_month.dt.year == year]
            .groupby(["ba_code", "report_date", "fuel_category_eia930"])[
                "net_generation_mwh"
            ]
            .sum()
            .reset_index()
        )
    return pd.concat(monthly, ignore_index=True)

Fit of OGE generation by BA and fuel with EIA-930.

In [ ]:
def compare_ba_generation_to_eia930(
    generation: pd.DataFrame,
    eia930_generation: pd.DataFrame,
    ba_col: str = "ba_code",
    min_eia930_generation_mwh: float = 10_000,
) -> pd.DataFrame:
    """Compares OGE net generation by BA and fuel category with EIA-930.

    Compares US BAs with at least `min_eia930_generation_mwh` of EIA-930 net generation
    in the year. Plants in BAs that don't report to EIA-930 are left out. All errors are
    relative to the BA's annual EIA-930 net generation:
    - `generation_error`: annual OGE minus EIA-930 net generation.
    - `fuel_mix_mismatch`: the sum over fuel categories of the absolute difference in
        annual net generation. It is 0 when every fuel matches, and is at least the
        absolute value of `generation_error`.
    - `monthly_generation_error` and `monthly_fuel_mix_mismatch`: the same, with the
        absolute differences summed over months, so that differences in timing within
        the year count too.

    Args:
        generation (pd.DataFrame): OGE monthly net generation, with "report_date",
            `ba_col`, "fuel_category_eia930" and "net_generation_mwh" columns.
        eia930_generation (pd.DataFrame): EIA-930 monthly net generation, from
            `aggregate_eia930_generation_by_month`.
        ba_col (str, optional): the column of `generation` with the BA of each
            generator. Defaults to "ba_code".
        min_eia930_generation_mwh (float, optional): BAs with less EIA-930 net
            generation in the year are left out. Defaults to 10,000 (10 GWh).

    Returns:
        pd.DataFrame: one row per BA, with its annual OGE and EIA-930 net generation
            and the errors above.
    """
    keys = ["ba_code", "report_date", "fuel_category_eia930"]
    ba_reference = load_data.load_ba_reference()
    us_bas = set(ba_reference.loc[ba_reference["us_ba"] == "Yes", "ba_code"])
    eia930_annual = eia930_generation.groupby("ba_code")["net_generation_mwh"].sum()
    bas = sorted(
        set(eia930_annual[eia930_annual >= min_eia930_generation_mwh].index) & us_bas
    )

    oge = (
        generation[
            [ba_col, "report_date", "fuel_category_eia930", "net_generation_mwh"]
        ]
        .rename(columns={ba_col: "ba_code"})
        .groupby(keys)["net_generation_mwh"]
        .sum()
        .rename("oge")
    )
    eia930 = eia930_generation.set_index(keys)["net_generation_mwh"].rename("eia930")
    comparison = pd.concat([oge, eia930], axis=1).fillna(0).reset_index()
    comparison = comparison[comparison["ba_code"].isin(bas)]
    comparison["difference"] = comparison["oge"] - comparison["eia930"]

    annual_by_fuel = comparison.groupby(["ba_code", "fuel_category_eia930"])[
        "difference"
    ].sum()
    monthly = comparison.groupby(["ba_code", "report_date"])["difference"].sum()
    ba_comparison = comparison.groupby("ba_code")[["oge", "eia930"]].sum()
    ba_comparison = ba_comparison.rename(
        columns={"oge": "oge_mwh", "eia930": "eia930_mwh"}
    )
    eia930_mwh = ba_comparison["eia930_mwh"]
    ba_comparison["generation_error"] = ba_comparison["oge_mwh"] / eia930_mwh - 1
    ba_comparison["fuel_mix_mismatch"] = (
        annual_by_fuel.abs().groupby("ba_code").sum() / eia930_mwh
    )
    ba_comparison["monthly_generation_error"] = (
        monthly.abs().groupby("ba_code").sum() / eia930_mwh
    )
    ba_comparison["monthly_fuel_mix_mismatch"] = (
        comparison["difference"].abs().groupby(comparison["ba_code"]).sum() / eia930_mwh
    )
    return ba_comparison.reset_index()


def summarize_ba_generation_comparison(ba_comparison: pd.DataFrame) -> pd.DataFrame:
    """Summarizes the errors of each BA from `compare_ba_generation_to_eia930`.

    Args:
        ba_comparison (pd.DataFrame): output of `compare_ba_generation_to_eia930`.

    Returns:
        pd.DataFrame: for each error, the number of BAs, the median and the EIA-930
            generation-weighted mean of its absolute value, and the number of BAs whose
            absolute error is above 10% and 20%.
    """
    summary = {}
    for metric in [
        "generation_error",
        "fuel_mix_mismatch",
        "monthly_generation_error",
        "monthly_fuel_mix_mismatch",
    ]:
        error = ba_comparison[metric].abs()
        summary[metric] = {
            "bas": len(error),
            "median": error.median(),
            "weighted_mean": np.average(error, weights=ba_comparison["eia930_mwh"]),
            "bas_above_10pct": (error > 0.1).sum(),
            "bas_above_20pct": (error > 0.2).sum(),
        }
    return pd.DataFrame(summary).T


def validate_ba_generation_against_eia930(
    year: int,
    ba_col: str = "ba_code",
    eia930_folder: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compares OGE net generation by BA and fuel category with reconciled EIA-930
    data, for all generation and for the generation that EIA-930A lists in Schedule 2.

    Reads the outputs of the data pipeline for `year`, including the cleaned EIA-930A
    tables (see `eia930a.clean_eia930a`).

    Args:
        year (int): a four-digit year.
        ba_col (str, optional): the column that assigns plants to BAs. Defaults to
            "ba_code".
        eia930_folder (str | None, optional): folder with the reconciled EIA-930 data.
            Defaults to `outputs/{year}/eia930/`.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: the comparison for each BA (see
            `compare_ba_generation_to_eia930`), and its summary (see
            `summarize_ba_generation_comparison`). Both have a "generation" column,
            which is "all" or "listed_in_eia930a".
    """
    if eia930_folder is None:
        eia930_folder = outputs_folder(f"{year}/eia930/")
    generation = load_generator_generation_for_ba_validation(year)
    eia930_generation = aggregate_eia930_generation_by_month(
        pd.read_parquet(
            f"{eia930_folder}/eia930_reconciled_net_generation_by_energy_source.parquet"
        ),
        year,
    )
    ba_comparisons = []
    summaries = []
    for name, subset in [
        ("all", generation),
        (
            "listed_in_eia930a",
            generation[generation["listed_in_eia930a"] != "unlisted"],
        ),
    ]:
        ba_comparison = compare_ba_generation_to_eia930(
            subset, eia930_generation, ba_col=ba_col
        )
        ba_comparisons.append(ba_comparison.assign(generation=name))
        summaries.append(
            summarize_ba_generation_comparison(ba_comparison)
            .rename_axis("metric")
            .reset_index()
            .assign(generation=name)
        )
    return (
        pd.concat(ba_comparisons, ignore_index=True),
        pd.concat(summaries, ignore_index=True),
    )

Map check: plants isolated from their own BA.

In [ ]:
def identify_plants_isolated_from_their_ba(
    plants: pd.DataFrame,
    ba_col: str = "ba_code",
    min_distance_km: float = 100,
    min_distance_ratio: float = 3,
) -> pd.DataFrame:
    """Identifies plants that are far from every other plant in their BA, and much
    closer to a plant in another BA, which may be assigned to the wrong BA.

    Counting the BAs of each plant's nearest neighbors flags many plants that are
    correctly assigned, e.g. where BAs are interleaved (Florida, the Carolinas) or
    large remote plants (Palo Verde, Grand Coulee). This test only flags plants that
    are isolated from their own BA.

    Args:
        plants (pd.DataFrame): table with "plant_id_eia", `ba_col`, "latitude" and
            "longitude" columns. Plants missing any of these are left out.
        ba_col (str, optional): the column that assigns plants to BAs. Defaults to
            "ba_code".
        min_distance_km (float, optional): a plant is isolated if the nearest plant in
            its own BA is further than this. Defaults to 100.
        min_distance_ratio (float, optional): and if that is at least this many times
            the distance to the nearest plant in another BA. Defaults to 3.

    Returns:
        pd.DataFrame: `plants` with the distance to the nearest plant in the same BA
            ("distance_to_own_ba_km"), the nearest plant in another BA
            ("nearest_other_ba" and "distance_to_other_ba_km"), and
            "isolated_from_ba". The distance to the own BA is infinite for the only
            plant in its BA, so such plants are always isolated.
    """
    earth_radius_km = 6371.0
    plants = plants.dropna(
        subset=["plant_id_eia", ba_col, "latitude", "longitude"]
    ).reset_index(drop=True)
    # points on a unit sphere, so that the straight-line distance between them can be
    # converted to the great-circle distance
    latitude = np.radians(plants["latitude"].to_numpy(dtype=float))
    longitude = np.radians(plants["longitude"].to_numpy(dtype=float))
    points = np.column_stack(
        [
            np.cos(latitude) * np.cos(longitude),
            np.cos(latitude) * np.sin(longitude),
            np.sin(latitude),
        ]
    )

    def to_km(chord):
        return 2 * earth_radius_km * np.arcsin(np.clip(chord / 2, 0, 1))

    distance_to_own_ba = np.full(len(plants), np.inf)
    distance_to_other_ba = np.full(len(plants), np.inf)
    nearest_other_ba = np.full(len(plants), None, dtype=object)
    bas = plants[ba_col].to_numpy()
    for ba in np.unique(bas):
        in_ba = np.flatnonzero(bas == ba)
        not_in_ba = np.flatnonzero(bas != ba)
        if len(in_ba) > 1:
            # the nearest plant is the plant itself
            chord, _ = cKDTree(points[in_ba]).query(points[in_ba], k=2)
            distance_to_own_ba[in_ba] = to_km(chord[:, 1])
        if len(not_in_ba) > 0:
            chord, index = cKDTree(points[not_in_ba]).query(points[in_ba], k=1)
            distance_to_other_ba[in_ba] = to_km(chord)
            nearest_other_ba[in_ba] = bas[not_in_ba[index]]

    plants["distance_to_own_ba_km"] = distance_to_own_ba
    plants["nearest_other_ba"] = nearest_other_ba
    plants["distance_to_other_ba_km"] = distance_to_other_ba
    plants["isolated_from_ba"] = (distance_to_own_ba > min_distance_km) & (
        distance_to_own_ba >= min_distance_ratio * distance_to_other_ba
    )
    return plants

## Fit with EIA-930

Errors are relative to each BA's annual EIA-930 net generation (see `compare_ba_generation_to_eia930` above):
- `generation_error`: OGE minus EIA-930 net generation.
- `fuel_mix_mismatch`: the sum over fuels of the absolute difference.
- `monthly_*`: the same, with absolute differences summed over months.

In [ ]:
ba_comparison, summary = validate_ba_generation_against_eia930(year, ba_col=ba_col)
summary.set_index(["generation", "metric"]).round(3)

In [ ]:
# BAs with the largest fuel mix mismatch
(
    ba_comparison[ba_comparison["generation"] == "all"]
    .sort_values("fuel_mix_mismatch", ascending=False)
    .head(30)
    .round(3)
)

## Map check

In [ ]:
generation = load_generator_generation_for_ba_validation(year)
plants = (
    generation.groupby(
        ["plant_id_eia", "plant_name_eia", ba_col, "state", "latitude", "longitude"]
    )["net_generation_mwh"]
    .sum()
    .reset_index()
)
plants = plants[plants["net_generation_mwh"] > 0]
plants = identify_plants_isolated_from_their_ba(plants, ba_col=ba_col)
isolated = plants[plants["isolated_from_ba"]].sort_values(
    "net_generation_mwh", ascending=False
)
print(f"{len(isolated)} of {len(plants)} plants are isolated from their BA")
isolated

In [ ]:
plants["size"] = plants["net_generation_mwh"].clip(lower=1000) ** 0.5
fig = px.scatter_geo(
    plants,
    lat="latitude",
    lon="longitude",
    color=ba_col,
    size="size",
    size_max=15,
    hover_name="plant_name_eia",
    hover_data=["plant_id_eia", ba_col, "nearest_other_ba", "net_generation_mwh"],
    scope="usa",
    opacity=0.6,
    title=f"Plants by {ba_col} in {year}, with isolated plants circled",
)
outliers = plants[plants["isolated_from_ba"]]
fig.add_scattergeo(
    lat=outliers["latitude"],
    lon=outliers["longitude"],
    text=outliers["plant_name_eia"] + " (" + outliers[ba_col] + ")",
    mode="markers",
    marker=dict(size=14, color="rgba(0,0,0,0)", line=dict(width=2, color="black")),
    name="isolated from BA",
)
fig.update_layout(height=700, showlegend=False)
fig.show()